In [1]:
import sys
sys.path.append("../src")

import torch
import torch.nn as nn
from data import generate_examples, generate_examples_with_error_type, parenthesis, make_edit_labels
from model import DyckTransformer

In [2]:
VOCAB = ["(", ")", "[", "]", "[PAD]", "[CLS]", "[SEP]"]
stoi = {tok: i for i, tok in enumerate(VOCAB)}
itos = {i: tok for tok, i in stoi.items()}
edit_stoi = {"OK": 0, "DELETE": 1, "INSERT())": 2, "INSERT(])": 3, "REPLACE())": 4, "REPLACE(])": 5}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
model = DyckTransformer(
    vocab_size=len(VOCAB),
    pad_idx=stoi["[PAD]"],
    num_edit_labels=6,
    d_model=128,
    n_heads=4,
    n_layers=2,
    max_len=80,
    dropout=0.1
)
model.load_state_dict(torch.load("../results/models/det_dyck_transformer.pt", map_location=device))
model.to(device)
model.eval()

DyckTransformer(
  (embed): Embedding(7, 128)
  (pos): SinusoidalPositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (cls_head): Linear(in_features=128, out_features=2, bias=True)
  (token_head): Linear(in_features=128, out_features=6, bias=True)
)

In [4]:
det_test_data_typed = generate_examples_with_error_type(5000, task="detection")
det_ood_test_data = generate_examples_with_error_type(5000, min_len=40, max_len=80, max_depth=7, task="detection")

In [5]:
model.eval()
from collections import defaultdict

results = defaultdict(lambda: {"preds": [], "labels": []})

batch_size = 64

with torch.no_grad():
    for i in range(0, len(det_test_data_typed), batch_size):
        batch = det_test_data_typed[i:i+batch_size]
        xs = [x for x, _, _ in batch]
        ys = [y for _, y, _ in batch]
        types = [t for _, _, t in batch]
        xs = torch.tensor([[stoi[t] for t in x] for x in xs]).to(device)
        ys_tensor = torch.tensor(ys).to(device)
        cls_logits, _ = model(xs)
        preds = torch.argmax(cls_logits, dim=1).cpu().tolist()
        for pred, label, etype in zip(preds, ys, types):
            results[etype]["preds"].append(pred)
            results[etype]["labels"].append(label)

for etype in ["none", "E1", "E2", "E3", "E4"]:
    preds = results[etype]["preds"]
    labels = results[etype]["labels"]
    acc = sum(p == l for p, l in zip(preds, labels)) / len(labels)
    f1 = f1_score(labels, preds, average="macro")
    print(f"{etype}: acc={acc:.4f} f1={f1:.4f} n={len(labels)}")

c:\Users\sarav\projects\dyck_transformer\.pixi\envs\default\Lib\site-packages\torch\nn\modules\transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\bld\libtorch_1772176602146\work\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


NameError: name 'f1_score' is not defined